In [ ]:
# A1 · Part 2 — Build a minimal retrieval system.  Starter notebook.
# PE6201 · Emerging AI Technologies. Run top to bottom, then replace the placeholder domain.

# A1 · Part 2 — Build a minimal retrieval system  *(starter)*

**What Part 2 asks of you.** Assemble a small collection of documents from **your** domain —
10 to 30 is enough. Build a system that retrieves relevant passages and uses them to answer
questions. Show it answering **five questions it could not possibly answer without your
documents**. Then **break it**: find a question where retrieval fails or returns the wrong
passage, and explain *why* it failed.

> **The breaking is not optional and it is not a penalty.** Finding the edge of your own system
> is the single most valuable thing you will do in this assignment. A notebook where everything
> works is a notebook that was not pushed hard enough.

**What this notebook is.** A working retrieval-augmented generation (RAG) pipeline on a
**placeholder domain** (houseplant care notes). Everything you must change is marked
**`### REPLACE ME`**.

**Cost.** About 10 short generation calls. Fractions of a cent. Embeddings run locally and free.

**Deliverable from this notebook:** this notebook (run, output visible), five worked examples,
one documented failure, and roughly 400 words.

> **If a cell does not work.** This notebook was written before you ran it, against library
> versions, model names and provider behaviour that change without notice — so treat it as a
> working draft, not gospel. **Read the error, then try to fix it yourself:** debugging is part of
> the work, not a detour from it. Then **report it** — in class or by email — with the cell, the
> full error text, and what you tried, so the shared copy gets fixed and nobody behind you loses
> the same hour. Reporting a problem costs you nothing. The one thing not to do is quietly work
> around a broken cell: if your fix changes what the notebook measures, say so in your writeup.

### Setup

In [1]:
%pip -q install openai scikit-learn sentence-transformers

### Connect to OpenRouter
The **generation** step (writing the answer) goes to OpenRouter with your key. The
**embedding** step (turning text into vectors) runs **locally and free** — OpenRouter serves chat
models, not embedding models, which is exactly the "which layer do you rent?" question from
Class 2.

In [2]:
import os, getpass, time
import numpy as np
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)
GEN_MODEL = 'openai/gpt-4o-mini'      # TRY THIS (knob): any model your key can reach
TOKENS = {'in': 0, 'out': 0}

def generate(prompt, system=None, max_new_tokens=300, temperature=0.0):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected ·', generate('Reply with exactly the word: ready', max_new_tokens=6))

connected · ready


### Check your remaining credit  —  free, this call is not billed
Your allocation is **fixed and will not be topped up**, so watch it rather than discovering it at
the end. `usage` is what you have spent; `limit` is your cap; the difference is what is left.
Run this at the start of a working session and again at the end.

In [3]:
import urllib.request, json as _json

def credit_check():
    """Ask OpenRouter what is left on your key. Costs nothing."""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.0043 of $10.00  ->  $9.9957 remaining
rate limit: -1 requests per 10s


---
## 0 · Five words you need — *start here if you don't code*

| Word | In plain words |
|---|---|
| **Embedding** | A piece of text turned into a list of numbers, arranged so that texts with similar *meaning* end up near each other. |
| **Chunk** | A small slice of a document. We search chunks, not whole documents, so what comes back is the relevant *paragraph* rather than the whole manual. |
| **Retrieval** | Finding the chunks nearest the question. Pure arithmetic — no model involved, no understanding. |
| **Top-k** | How many chunks we fetch. Too few and the answer is missing; too many and the real answer drowns in noise. |
| **Grounding** | Making the model answer **only** from the retrieved text, and say so when the text does not cover it. |

**RAG in one line:** *embed your documents → retrieve the ones nearest the question → let a model
answer from them.* **The model is rented; the retrieval is yours.** That is why retrieval is where
almost every failure lives — and why the failure you document is the interesting part.

---
## 1 · Your documents  ### REPLACE ME

**10 to 30 documents.** Each a short paragraph — a few sentences is fine.

**The one rule that matters:** they must contain things a model *could not already know*. If your
documents are summaries of Wikipedia, the model answers correctly with retrieval switched off, and
you have proved nothing. Good sources: your own notes, internal procedures, club or team records,
a rulebook nobody has digitised, your family's version of anything, recent local information,
specifics with names and numbers in them.

**Test for this later, in Section 4.** We ask the model each question with the documents removed.
If it still gets them right, your document set is too generic — go back and make it specific.

In [31]:
### REPLACE ME — your documents. 10-30 short paragraphs from YOUR domain.
DOCS = [
    "# iPhone Product & Specifications Guide\n## Product Overview\nThis guide provides technical specifications, operational rules, and hardware limits for current flagship iPhone models (including iPhone 15, iPhone 15 Pro, and Pro Max variants).\n## Hardware Specifications\n- **Processors**: iPhone 15 uses the A16 Bionic chip. iPhone 15 Pro and Pro Max feature the A17 Pro chip built on a 3nm process.\n- **Display**: All models feature Super Retina XDR displays with Dynamic Island. Pro models include ProMotion technology (1Hz to 120Hz refresh rates) and Always‑On display capability.\n- **Battery Life**: \n  - iPhone 15: Up to 20 hours of video playback.\n  - iPhone 15 Pro: Up to 23 hours of video playback.\n  - iPhone 15 Pro Max: Up to **24 hours** of video playback.\n- **Charging & Port**: USB‑C connector on all models. iPhone 15 supports USB 2 speeds (up to 480Mbps). Pro models support USB 3 speeds (up to 10Gbps) when using a compatible 10Gbps cable.\n## Camera System\n- **iPhone 15**: Dual‑camera system featuring a 48MP Main sensor and 12MP Ultra Wide sensor, supporting 2x optical quality telephoto zoom.\n- **iPhone 15 Pro / Pro Max**: Triple‑camera system with a 48MP Main sensor, 12MP Ultra Wide sensor, and Telephoto lens. \n  - iPhone 15 Pro offers 3x optical zoom.\n  - iPhone 15 Pro Max offers 5x optical zoom (120mm equivalent) via a tetraprism design.",
    "# iPad Product & Compatibility Guide\n## Lineup Overview\nThe iPad lineup consists of four distinct tiers: iPad (Base), iPad mini, iPad Air, and iPad Pro.\n## Chipset & Display Differences\n- **iPad (10th Generation)**: Powered by A14 Bionic. Liquid Retina display (60Hz), non‑laminated screen without anti‑reflective coating.\n- **iPad Air**: Powered by Apple M2 chip. 11‑inch and 13‑inch Liquid Retina displays with P3 wide color and anti‑reflective coating.\n- **iPad Pro**: Powered by Apple M4 chip. Tandem OLED Ultra Retina XDR display (ProMotion 10Hz‑120Hz). Available in 11‑inch and 13‑inch options.\n## Apple Pencil Compatibility Rules\n- **Apple Pencil (USB‑C)**: Compatible with all USB‑C iPad models (iPad 10th gen, iPad mini 6th gen, iPad Air M2, iPad Pro M4).\n- **Apple Pencil (2nd Generation)**: Compatible ONLY with older iPad Air (4th/5th gen), iPad mini 6th gen, and previous‑gen iPad Pro models. **NOT compatible with iPad Pro M4 or iPad Air M2.**\n- **Apple Pencil Pro**: Compatible ONLY with iPad Air (M2) and iPad Pro (M4). Features barrel roll, squeeze gestures, and haptic feedback.",
    "# MacBook Lineup Technical Reference\n## Product Models\nThis document covers MacBook Air (13‑inch and 15‑inch) and MacBook Pro (14‑inch and 16‑inch) configurations.\n## Architecture & Memory Specs\n- **MacBook Air (M3)**: Fanless design. Supports up to 24GB Unified Memory. Drives up to two external displays only when the laptop lid is closed.\n- **MacBook Pro (M3 / M3 Pro / M3 Max)**: Active thermal cooling system. \n  - M3 supports up to 24GB Unified Memory.\n  - M3 Pro supports up to 36GB Unified Memory and up to two external displays.\n  - M3 Max supports up to **96GB** Unified Memory and up to four external displays.\n## Battery & Expansion Ports\n- **MacBook Air**: MagSafe 3 charging port, 2x Thunderbolt / USB 4 ports, 3.5mm headphone jack. Battery life up to 18 hours.\n- **MacBook Pro**: MagSafe 3, 3x Thunderbolt 4 ports (2x on base M3), HDMI port, SDXC card slot, 3.5mm headphone jack. Battery life up to 22 hours on 16‑inch models.",
    "# AirPods Product & Audio Features Guide\n## Model Lineup\nCovers AirPods (3rd Generation), AirPods Pro (2nd Generation with USB‑C), and AirPods Max.\n## Active Noise Cancellation (ANC) & Audio Technologies\n- **AirPods (3rd Gen)**: Open‑ear design. Features Personalized Spatial Audio with dynamic head tracking. **Does NOT have Active Noise Cancellation (ANC) or Transparency mode.**\n- **AirPods Pro (2nd Gen)**: In‑ear design with silicone tips. Features up to 2x more Active Noise Cancellation, Adaptive Audio, Transparency mode, and Conversation Awareness. Powered by H2 chip.\n- **AirPods Max**: Over‑ear headphones. Features ANC, Transparency mode, and Digital Crown control. Powered by dual H1 chips.\n## Battery & Case Features\n- **AirPods Pro (2nd Gen)**: 6 hours listening time on a single charge (with ANC enabled). Case includes MagSafe charging, Precision Finding (U1 chip), built‑in speaker for Find My alerts, and a lanyard loop.",
    "# Apple Watch Features & Water Resistance Guide\n## Model Comparison\n- **Apple Watch Series 9**: Available in Aluminum and Stainless Steel cases (41mm and 45mm). Display up to 2000 nits. Features S9 SiP and Double Tap gesture.\n- **Apple Watch Ultra 2**: Titanium case (49mm). Display up to 3000 nits. Dual‑frequency GPS (L1 and L5), customizable Action Button, and built‑in siren.\n## Water Resistance & Depth Limits\n- **Series 9**: Water resistant to 50 meters under ISO standard 22810:2010. Suitable for shallow‑water activities like swimming in a pool or ocean. **Not suitable for scuba diving, waterskiing, or high‑velocity water activities.**\n- **Ultra 2**: Water resistant to **85 meters** under ISO standard 22810. Certified to EN13319 for recreational scuba diving to 40 meters and high‑speed watersports.\n## Health Sensors\nBoth models feature ECG, blood oxygen measurements, temperature sensing for ovulation tracking, and Fall Detection. Blood oxygen functionality is subject to regional availability and legal regulations.",
    "# Standard Return & Refund Policy\n## Return Window\nCustomers have **14 calendar days** from the date of receiving the product to initiate a return for any item purchased directly from the official store.\n## Eligibility & Condition Requirements\n- Items must be returned in their original condition with all original packaging, cables, adapters, and manuals included.\n- Products showing physical damage, liquid damage, or unauthorized modification are ineligible for return.\n- Open box items are eligible for a full refund as long as all contents remain intact.\n## Refund Processing Times\n- Once received at the warehouse, inspection takes **2 to 3 business days**.\n- Refund payment will be issued back to the original payment method. Credit card refunds typically take **3 to 5 business days** to post after approval.\n- Original expedited shipping fees are non‑refundable.",
    "# Apple Trade In Program Guidelines\n## Trade In Process\n1. Customers provide device details (serial number, model, condition) online to receive an estimated trade‑in value.\n2. If accepting the offer during a new purchase, an instant credit is applied, or a trade‑in kit is dispatched to the customer's shipping address.\n3. Customer must mail back the trade‑in device within **11 calendar days** of receiving the new product.\n## Inspection & Re‑valuation Rules\n- Returned devices undergo automated and physical inspection upon arrival at the processing facility.\n- If the device condition differs from described (e.g., unreported screen cracks, liquid damage, third‑party battery replacement), a **revised valuation quote** is sent via email.\n- The customer has **7 calendar days** to accept or decline the revised quote. If declined, the device is shipped back free of charge.\n## Payment & Credit Limits\n- Instant store credit can only be applied toward a new hardware purchase.\n- If the trade‑in value exceeds the new product price, the balance is issued via an Apple Gift Card.\n- **Maximum 2 trade‑in device can be applied online per transaction.** Multiple trade‑in devices must be processed separately in‑store.",
    "# Delivery & Shipping Guidelines\n## Shipping Options & Timelines\n- **Standard Shipping**: Free on all eligible orders. Delivery within **2 to 4 business days** after order processing.\n- **Courier Express Delivery**: Available in select metropolitan areas for a fixed fee. Delivery within 2 hours or scheduled 2‑hour windows.\n- **Pickup at Apple Store**: Free option. Available within 2 hours for in‑stock items after order placement.\n## Signature Requirements & Address Changes\n- High‑value orders (exceeding $750 value, including MacBooks, iPhones, and iPads) **require a direct signature** upon delivery.\n- Address modifications cannot be made once an order status has changed to \"Preparing for Shipment\" or \"Shipped\".\n- If delivery fails after 3 attempts, the shipment is returned to the warehouse and automatically processed as a return refund.",
    "# Promotional Stacking & Higher Education Discount Rules\n## Higher Education Discount (Back to School)\n- Eligible buyers: Higher education students, parents buying for college students, teachers, and staff at all levels.\n- Provides specialized discounted pricing on eligible Mac and iPad models.\n## Promotion Stacking Regulations\n- **Trade‑In Stacking**: Education pricing CAN be combined with an Apple Trade In offer on the same transaction.\n- **Gift Card / Accessory Promotions**: Seasonal bundle offers (e.g., free AirPods or Apple Gift Card with Mac purchase) CANNOT be combined with non‑educational corporate promotional codes.\n- **Quantity Limits**: Qualified purchasers are limited to purchasing **one (1) desktop Mac, one (1) Mac mini, one (1) notebook Mac, and two (2) iPads** per academic year under education pricing.\n## Bundle Partial Returns\nIf a customer returns the main product (e.g., MacBook) while keeping the promotional bundle item (e.g., free gift card or discounted AirPods), the full retail value of the promotional item will be deducted from the total refund amount.",
    "# E‑commerce Product Listing & Compliance Regulations\n## Listing Copy Guidelines\n- Product descriptions must strictly adhere to official technical specifications. Operators must never quote unauthorized third‑party benchmarks or unverified battery estimates.\n- Claims such as \"Waterproof\" are strictly prohibited; copy must specify exact depth and rating (e.g., \"Water resistant under ISO standard 22810:2010\").\n## Warranty & Service Disclaimers\n- Product detail pages must display standard warranty disclaimers: \"Includes 1‑year limited warranty and up to 90 days of complimentary technical support.\"\n- Extended warranty claims must explicitly state \"AppleCare+ sold separately.\"\n- Operators must not promise lifetime replacements, free battery changes after standard warranty, or non‑certified repair service options under any operational circumstance."
]

print(f'{len(DOCS)} documents')
assert 10 <= len(DOCS) <= 40, 'The brief asks for 10-30 documents.'

10 documents


### Chunking
Long documents get cut into overlapping slices before embedding. Why: one embedding holds one
*average* meaning, so a long multi-topic document becomes a blurry vector that matches everything
a little and nothing well. Small chunks keep each vector about a single idea. The **overlap** stops
an idea that straddles a boundary from being cut in half.

This is a knob you tune against your evidence, not a setting with a correct value.

In [32]:
# TRY THIS (knob): watch what changes in Section 3 when you move these.
CHUNK_WORDS = 100     # try 15 (tiny) or 300 (whole document = one chunk)
OVERLAP     = 10     # words shared with the next chunk

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words, out, i = text.split(), [], 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = [(c, d_i) for d_i, d in enumerate(DOCS) for c in chunk(d)]
print(f'{len(DOCS)} docs -> {len(CHUNKS)} chunks (CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP})')

10 docs -> 22 chunks (CHUNK_WORDS=100, OVERLAP=10)


### Embed and retrieve
The embedder runs locally — a small model downloaded once, no key, no cost. If the download fails
(no internet on the machine, or a locked-down environment), the fallback cell below uses **TF-IDF**
word-overlap instead. TF-IDF works, and it is a useful contrast: it matches *words*, not *meaning*,
so it fails on paraphrase. If you use the fallback, say so in your writeup — it changes which
failures you find.

In [40]:
EMBEDDER = 'sentence-transformers/all-MiniLM-L6-v2'
USING = None
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(EMBEDDER)
    def embed(texts):
        v = _m.encode(list(texts), normalize_embeddings=True)
        return np.asarray(v)
    USING = f'local embeddings ({EMBEDDER}) — matches MEANING'
except Exception as e:
    print('embedder unavailable ->', str(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    _v = TfidfVectorizer().fit([c for c, _ in CHUNKS])
    def embed(texts):
        m = _v.transform(list(texts)).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.where(n == 0, 1, n)
    USING = 'TF-IDF fallback — matches WORDS, not meaning'

print('using:', USING)
MATRIX = embed([c for c, _ in CHUNKS])

TOP_K = 1            # TRY THIS (knob): 1 (brittle) or 8 (noisy)

def retrieve(question, k=TOP_K):
    q = embed([question])[0]
    scores = MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], float(scores[i])) for i in order]

for txt, d_i, s in retrieve('how often is the fiddle leaf fig watered in winter?'):
    print(f'  {s:.3f}  [doc {d_i}]  {txt[:80]}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

using: local embeddings (sentence-transformers/all-MiniLM-L6-v2) — matches MEANING
  0.172  [doc 5]  # Standard Return & Refund Policy ## Return Window Customers have **14 calendar ...


### Answer from the retrieved text — and only from it
The instruction to say *"the notes do not say"* is the whole point. Without it the model fills gaps
from its training data and you cannot tell which sentences came from your documents. That is
**silent failure**: a confident, fluent, unsourced answer.

In [41]:
GROUNDED = (
    "Answer using ONLY the notes provided. "
    "If the notes do not contain the answer, reply exactly: The notes do not say. "
    "Do not use any other knowledge. Quote the phrase you relied on."
)

def answer(question, k=TOP_K, show=True):
    hits = retrieve(question, k)
    context = '\n\n'.join(f'[{i+1}] {t}' for i, (t, _, _) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {question}', system=GROUNDED)
    if show:
        print(f'Q: {question}\nA: {out}\n   (top chunk score {hits[0][2]:.3f}, doc {hits[0][1]})\n')
    return out, hits

answer('When was the fiddle-leaf fig repotted, and into what?')

Q: When was the fiddle-leaf fig repotted, and into what?
A: The notes do not say.
   (top chunk score 0.114, doc 5)



('The notes do not say.',
 [('# Standard Return & Refund Policy ## Return Window Customers have **14 calendar days** from the date of receiving the product to initiate a return for any item purchased directly from the official store. ## Eligibility & Condition Requirements - Items must be returned in their original condition with all original packaging, cables, adapters, and manuals included. - Products showing physical damage, liquid damage, or unauthorized modification are ineligible for return. - Open box items are eligible for a full refund as long as all contents remain intact. ## Refund Processing Times - Once received at the warehouse, inspection takes',
   5,
   0.11425148695707321)])

---
## 2 · Five questions it could not answer without your documents  ### REPLACE ME

Each question should have a **specific** answer that lives in your documents — a date, a name, a
number, a decision and its outcome. "What is a monstera?" is a bad question: the model knows that
already, so a correct answer proves nothing about your retrieval.

In [38]:
### REPLACE ME — five questions answerable ONLY from your documents.
QUESTIONS = [
    "What is the maximum number of trade‑in devices permitted in one online Apple trade‑in transaction?",
    "Within how many calendar days must a customer return their trade‑in device after receiving the new product?",
    "What is the maximum unified memory available for the MacBook Pro M3 Max?",
    "How many hours of video playback can the iPhone 15 Pro Max deliver?",
    "What is the ISO‑standard water‑resistance depth rating of the Apple Watch Ultra 2?",
]
for q in QUESTIONS:
    answer(q)

Q: What is the maximum number of trade‑in devices permitted in one online Apple trade‑in transaction?
A: The maximum number of trade‑in devices permitted in one online Apple trade‑in transaction is "2."
   (top chunk score 0.714, doc 6)

Q: Within how many calendar days must a customer return their trade‑in device after receiving the new product?
A: Customers must mail back the trade‑in device within **11 calendar days** of receiving the new product.
   (top chunk score 0.674, doc 6)

Q: What is the maximum unified memory available for the MacBook Pro M3 Max?
A: The maximum unified memory available for the MacBook Pro M3 Max is **96GB**. "M3 Max supports up to **96GB** Unified Memory and"
   (top chunk score 0.702, doc 2)

Q: How many hours of video playback can the iPhone 15 Pro Max deliver?
A: The notes say: "iPhone 15 Pro Max: Up to **24 hours** of video playback."
   (top chunk score 0.744, doc 0)

Q: What is the ISO‑standard water‑resistance depth rating of the Apple Watch Ultra 2

---
## 3 · Break it  ### REPLACE ME

**This section carries the marks.** You are looking for a question where retrieval brings back the
wrong passage — not where the model writes a clumsy sentence.

Four kinds of break that reliably work. Try all four; keep the one that fails most interestingly.

| Kind | What to try |
|---|---|
| **Vocabulary mismatch** | Ask using a word that never appears in your documents but means the same thing. Devastating for TF-IDF; embeddings usually survive it. |
| **Two documents needed** | Ask something that requires combining facts from two documents. Top-k retrieval fetches the *most similar* chunks, which is not the same as *the set that answers the question*. |
| **Distractor** | Ask about a topic covered in several documents where only one is relevant. Watch the right chunk get outvoted. |
| **Absent but plausible** | Ask something your documents nearly, but do not quite, cover. Does it say "the notes do not say", or does it invent? |

**Diagnose, do not just report.** "It got it wrong" is a Developing answer. "The right chunk ranked
fourth with score 0.31 while an irrelevant chunk scored 0.44 because both contain the word
*water*" is a Proficient one.

In [42]:
### REPLACE ME — your breaking attempts.
BREAKERS = [
    ("vocabulary mismatch", "What maximum amount of onboard RAM can the highest‑spec M3 laptop support?"),
    ("two documents needed", "Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?"),
    ("distractor", "Which Apple Watch model is certified for recreational scuba diving and what is its general ISO water‑resistance rating?"),
    ("absent but plausible", "How much does Apple charge for the AppleCare+ extended warranty for iPhone 15 Pro Max?"),
]
for kind, q in BREAKERS:
    print(f'--- {kind.upper()} ---')
    out, hits = answer(q)
    print('   retrieved:')
    for t, d_i, s in hits:
        print(f'     {s:.3f} [doc {d_i}] {t[:70]}...')
    print()

--- VOCABULARY MISMATCH ---
Q: What maximum amount of onboard RAM can the highest‑spec M3 laptop support?
A: The notes do not say.
   (top chunk score 0.633, doc 2)

   retrieved:
     0.633 [doc 2] # MacBook Lineup Technical Reference ## Product Models This document c...

--- TWO DOCUMENTS NEEDED ---
Q: Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?
A: Yes, you can combine student education pricing with an Apple trade-in offer on the same transaction. The notes do not say about the trade-in device limit per single online transaction.
   (top chunk score 0.695, doc 8)

   retrieved:
     0.695 [doc 8] # Promotional Stacking & Higher Education Discount Rules ## Higher Edu...

--- DISTRACTOR ---
Q: Which Apple Watch model is certified for recreational scuba diving and what is its general ISO water‑resistance rating?
A: The notes do not say.
   (top chunk score 0.777, doc 4)

   retrieved:
     0.777

### Diagnose the one you will submit
Fill this in **in prose**, from the scores printed above. This is your documented failure.

In [ ]:
FAILURE_DIAGNOSIS = """
### 
Question that failed:
Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?

What the system answered:
Yes, you can combine student education pricing with an Apple trade‑in offer on the same transaction. The notes do not say about the trade‑in device limit per single online transaction.

What the correct answer was:
Student education pricing can be combined with Apple trade‑in on the same online purchase, and a maximum of one trade‑in device can be applied per single online transaction.

Which chunk SHOULD have been retrieved, and what did it score:
The chunk containing trade‑in device limit rule from document 6. This required chunk was never retrieved because top‑k is set to 1.

What was retrieved instead, and what did it score:
Top‑1 retrieved chunk (score 0.695 from doc 8). This passage only explains that education pricing can stack with trade‑in, and contains no information regarding how many trade‑in devices are allowed for one online order. Only this single chunk is passed into the LLM context.

WHY (the mechanism — vocabulary, chunk size, top‑k, a distractor, or genuinely absent):
Two‑documents‑needed failure. The question depends on separate facts stored inside two different source documents. Under top‑k=1 setting, retrieval selects only the highest‑similarity chunk. The second necessary fact from another document is excluded entirely from context. The LLM can answer the first half of the question but has no access to the second constraint. Small chunk size further separates facts into more fragments and increases this failure chance.

What you would change to fix it, and what that change would cost you elsewhere:
Increase top‑k value so multiple chunks from different documents can be included in prompt context. The cost is larger prompt size, higher token consumption, longer inference time, and greater risk of injecting irrelevant distracting content into the context window.
"""
print(FAILURE_DIAGNOSIS)



### 
Question that failed:
Can I combine student education pricing with Apple trade‑in online, and what is the trade‑in device limit per single online transaction?

What the system answered:
Yes, you can combine student education pricing with an Apple trade‑in offer on the same transaction. The notes do not say about the trade‑in device limit per single online transaction.

What the correct answer was:
Student education pricing can be combined with Apple trade‑in on the same online purchase, and a maximum of one trade‑in device can be applied per single online transaction.

Which chunk SHOULD have been retrieved, and what did it score:
The chunk containing trade‑in device limit rule from document 6. This required chunk was never retrieved because top‑k is set to 1.

What was retrieved instead, and what did it score:
Top‑1 retrieved chunk (score 0.695 from doc 8). This passage only explains that education pricing can stack with trade‑in, and contains no information regarding how many t

---
## 4 · Two checks worth running before you write

**Check 1 — is retrieval doing anything?** Ask the same five questions with the documents removed.
If the model still answers correctly, your documents are too generic and Part 2 has not been
demonstrated. Fix the documents.

**Check 2 — does the grounding instruction hold?** A system that invents when it should decline is
worse than one that declines too often, because you cannot see it happening.

In [39]:
print('=== CHECK 1: same questions, NO documents ===')
for q in QUESTIONS[:5]:
    print(f'Q: {q}\nA: {generate(q, system="Answer in one short sentence.", max_new_tokens=80)}\n')

print('=== CHECK 2: something your documents definitely do not cover ===')
answer('What is the postal code of the flat?')

=== CHECK 1: same questions, NO documents ===
Q: What is the maximum number of trade‑in devices permitted in one online Apple trade‑in transaction?
A: The maximum number of trade-in devices permitted in one online Apple trade-in transaction is five.

Q: Within how many calendar days must a customer return their trade‑in device after receiving the new product?
A: A customer must return their trade-in device within 14 calendar days after receiving the new product.

Q: What is the maximum unified memory available for the MacBook Pro M3 Max?
A: The maximum unified memory available for the MacBook Pro M3 Max is 128GB.

Q: How many hours of video playback can the iPhone 15 Pro Max deliver?
A: The iPhone 15 Pro Max can deliver up to 29 hours of video playback.

Q: What is the ISO‑standard water‑resistance depth rating of the Apple Watch Ultra 2?
A: The Apple Watch Ultra 2 has an ISO-standard water-resistance depth rating of 100 meters.

=== CHECK 2: something your documents definitely do not 

('The notes do not say.',
 [('# Delivery & Shipping Guidelines ## Shipping Options & Timelines - **Standard Shipping**: Free on all eligible orders. Delivery within **2 to 4 business days** after order processing. - **Courier Express Delivery**: Available in select metropolitan areas for a fixed fee. Delivery within 2 hours or scheduled 2‑hour windows. - **Pickup at Apple Store**: Free option. Available within 2 hours for in‑stock items after order placement. ## Signature Requirements & Address Changes - High‑value orders (exceeding $750 value, including MacBooks, iPhones, and iPads) **require a direct signature** upon delivery. - Address modifications cannot be made once an order status has',
   7,
   0.3069677948951721),
  ('Address modifications cannot be made once an order status has changed to "Preparing for Shipment" or "Shipped". - If delivery fails after 3 attempts, the shipment is returned to the warehouse and automatically processed as a return refund.',
   7,
   0.1865818798

---
## 5 · YOUR TURN

1. Replace `DOCS` with 10–30 documents from your domain that contain things a model cannot know.
2. Replace `QUESTIONS` with five that are answerable only from them. Run and read every answer.
3. Run Check 1. If the model answers without your documents, go back to step 1.
4. Work through all four `BREAKERS`. Keep the most interesting failure and diagnose it properly.
5. Try `CHUNK_WORDS`, `OVERLAP` and `TOP_K` at extremes. Note which failures appear and disappear.

### What your ~400 words must contain

- What your documents are, and why a model could not answer these questions without them.
- The five worked examples (the notebook output is enough; the words are for what they show).
- **The failure, diagnosed** — which chunk should have won, which did, the scores, and the
  mechanism.
- What you would change to fix it, and **what that fix would break** — bigger chunks improve
  context and dilute matching; a larger top-k improves recall and adds noise and tokens. There is
  no free setting, and saying so is the postgraduate move.

In [43]:
print(f'tokens this session: {TOKENS["in"]} in / {TOKENS["out"]} out')

tokens this session: 13642 in / 1095 out
